# Busca em profundidade — Tutorial

**Algoritmos e Estruturas de Dados 2 — DCOMP/UFS**
Prof. Dr. André Yoshiaki Kashiwabara

Este tutorial acompanha a aula e segue de perto o material
[*Algoritmos para Grafos (em linguagem C)*](https://www.ime.usp.br/~pf/algoritmos_para_grafos/),
de Paulo Feofiloff (IME-USP), no capítulo
[Busca em profundidade](https://www.ime.usp.br/~pf/algoritmos_para_grafos/aulas/dfs.html).

## Objetivos

Ao final deste tutorial você será capaz de:

- implementar a busca em profundidade recursiva nas duas representações de grafo;
- explicar por que o vetor `pre[]` marca *e* numera os vértices;
- escrever a varredura `GRAPHdfs()` e enxergar a floresta DFS nos vetores `pre[]` e `pai[]`;
- reescrever a busca com uma pilha explícita e explicar por que a ordem de visita muda;
- medir o custo da busca em cada representação;
- recuperar um caminho de `s` a `t` a partir do vetor `pai[]`.

## Como este notebook funciona

O kernel é Python 3, mas **todo o código do tutorial é C**. Cada exemplo é
gravado em um arquivo `.c` com a mágica `%%writefile` e depois compilado e
executado com `gcc`. Execute as células na ordem.

Os exemplos usam o grafo da aula: vértices $0..5$ e os arcos
`0-1  0-5  1-0  1-5  2-4  3-1  5-3`.

In [1]:
# Preparação do ambiente: cria a pasta de trabalho e confere o compilador.
import os, subprocess, textwrap

os.makedirs('src', exist_ok=True)
print(subprocess.run(['gcc', '--version'], capture_output=True, text=True).stdout.splitlines()[0])

def compilar_e_rodar(fonte, entrada=None):
    """Compila src/<fonte> com gcc (C99) e executa, mostrando a saída."""
    exe = os.path.join('src', os.path.splitext(fonte)[0])
    c = subprocess.run(['gcc', '-std=c99', '-Wall', os.path.join('src', fonte), '-o', exe],
                       capture_output=True, text=True)
    if c.returncode != 0:
        print('ERRO DE COMPILACAO:\n' + c.stderr)
        return
    if c.stderr:
        print('avisos do compilador:\n' + c.stderr)
    r = subprocess.run([exe], capture_output=True, text=True, input=entrada)
    print(r.stdout, end='')
    if r.stderr:
        print('stderr:', r.stderr)

Apple clang version 21.0.0 (clang-2100.1.1.101)


## 1. DFS recursiva sobre a matriz de adjacências

A busca em profundidade a partir de um vértice `s` faz uma coisa só: marca o
vértice em que está e avança para o primeiro vizinho ainda não marcado. Quando
não há mais vizinhos novos, a recursão devolve o controle a quem chamou — é o
"volte por onde veio".

O vetor `pre[]` acumula os dois papéis: `pre[v] == -1` significa "ainda não
visitado" e, depois da visita, `pre[v]` guarda a **ordem de descoberta** de `v`.

Percorrendo a linha `v` da matriz da esquerda para a direita, os vizinhos são
examinados em **ordem crescente** — é o que torna a simulação da aula
reproduzível.

In [2]:
%%writefile src/dfs_matriz.c
/* BUSCA EM PROFUNDIDADE - representacao por matriz de adjacencias.
   Adaptado de P. Feofiloff, "Algoritmos para Grafos (em linguagem C)", IME-USP.
   https://www.ime.usp.br/~pf/algoritmos_para_grafos/aulas/dfs.html */
#include <stdio.h>
#include <stdlib.h>

#define vertex int
#define maxV 100

struct graph {
   int V;      /* numero de vertices */
   int A;      /* numero de arcos    */
   int **adj;  /* matriz de adjacencias */
};
typedef struct graph *Graph;

static int **MATRIXint( int r, int c, int val) {
   int **m = malloc( r * sizeof (int *));
   for (vertex i = 0; i < r; ++i) m[i] = malloc( c * sizeof (int));
   for (vertex i = 0; i < r; ++i)
      for (vertex j = 0; j < c; ++j) m[i][j] = val;
   return m;
}
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V; G->A = 0; G->adj = MATRIXint( V, V, 0);
   return G;
}
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   if (G->adj[v][w] == 0) { G->adj[v][w] = 1; G->A++; }
}

static int cnt;          /* contador de ordem de visita */
static int pre[maxV];    /* pre[v] = ordem de visita de v; -1 = nao visitado */

/* Visita todos os vertices ainda nao visitados que sao alcancaveis a
   partir de v. */
static void dfsR( Graph G, vertex v) {
   printf( "  visita %d  (pre = %d)\n", v, cnt);
   pre[v] = cnt++;
   for (vertex w = 0; w < G->V; ++w)
      if (G->adj[v][w] != 0)
         if (pre[w] == -1)
            dfsR( G, w);
   printf( "  fim de dfsR(%d)\n", v);
}

/* Faz uma unica busca, com raiz s, e mostra o resultado. */
void GRAPHdfsFrom( Graph G, vertex s) {
   for (vertex v = 0; v < G->V; ++v) pre[v] = -1;
   cnt = 0;
   printf( "busca em profundidade a partir de %d:\n", s);
   dfsR( G, s);

   printf( "\npre[] =");
   for (vertex v = 0; v < G->V; ++v) printf( " %2d", pre[v]);
   printf( "\nalcancaveis a partir de %d:", s);
   for (vertex v = 0; v < G->V; ++v) if (pre[v] != -1) printf( " %d", v);
   printf( "\nnao alcancaveis        :");
   for (vertex v = 0; v < G->V; ++v) if (pre[v] == -1) printf( " %d", v);
   printf( "\n");
}

int main( void) {
   Graph G = GRAPHinit( 6);
   GRAPHinsertArc( G, 0, 1); GRAPHinsertArc( G, 0, 5);
   GRAPHinsertArc( G, 1, 0); GRAPHinsertArc( G, 1, 5);
   GRAPHinsertArc( G, 2, 4); GRAPHinsertArc( G, 3, 1);
   GRAPHinsertArc( G, 5, 3);

   GRAPHdfsFrom( G, 0);
   return 0;
}

Overwriting src/dfs_matriz.c


In [3]:
compilar_e_rodar('dfs_matriz.c')
# Saida esperada: ordem de visita 0, 1, 5, 3; pre[] = 0 1 -1 3 -1 2;
# os vertices 2 e 4 NAO sao alcancaveis a partir de 0.

busca em profundidade a partir de 0:
  visita 0  (pre = 0)
  visita 1  (pre = 1)
  visita 5  (pre = 2)
  visita 3  (pre = 3)
  fim de dfsR(3)
  fim de dfsR(5)
  fim de dfsR(1)
  fim de dfsR(0)

pre[] =  0  1 -1  3 -1  2
alcancaveis a partir de 0: 0 1 3 5
nao alcancaveis        : 2 4


## 2. A mesma busca sobre listas de adjacência

O corpo do algoritmo não muda: o que muda é **como** enumeramos os vizinhos.
Em vez de varrer a linha `v` da matriz, percorremos a lista `G->adj[v]`.

Repare no efeito colateral: `GRAPHinsertArc()` insere cada nó **no início** da
lista, então a lista de `v` fica na ordem *inversa* à de inserção dos arcos. A
busca continua correta — o conjunto de vértices alcançados é o mesmo —, mas a
ordem de visita muda de `0, 1, 5, 3` para `0, 5, 3, 1`.

In [4]:
%%writefile src/dfs_listas.c
/* BUSCA EM PROFUNDIDADE - representacao por listas de adjacencia. */
#include <stdio.h>
#include <stdlib.h>

#define vertex int
#define maxV 100

typedef struct node *link;
struct node { vertex w; link next; };
struct graph { int V; int A; link *adj; };
typedef struct graph *Graph;

static link NEWnode( vertex w, link next) {
   link a = malloc( sizeof (struct node));
   a->w = w; a->next = next; return a;
}
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V; G->A = 0;
   G->adj = malloc( V * sizeof (link));
   for (vertex v = 0; v < V; ++v) G->adj[v] = NULL;
   return G;
}
/* Insere o arco v-w no INICIO da lista de v. */
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   for (link a = G->adj[v]; a != NULL; a = a->next)
      if (a->w == w) return;
   G->adj[v] = NEWnode( w, G->adj[v]); G->A++;
}
void GRAPHshow( Graph G) {
   for (vertex v = 0; v < G->V; ++v) {
      printf( "%2d:", v);
      for (link a = G->adj[v]; a != NULL; a = a->next) printf( " %2d", a->w);
      printf( "\n");
   }
}

static int cnt;
static int pre[maxV];

static void dfsR( Graph G, vertex v) {
   pre[v] = cnt++;
   for (link a = G->adj[v]; a != NULL; a = a->next) {
      vertex w = a->w;
      if (pre[w] == -1)
         dfsR( G, w);
   }
}

int main( void) {
   Graph G = GRAPHinit( 6);
   GRAPHinsertArc( G, 0, 1); GRAPHinsertArc( G, 0, 5);
   GRAPHinsertArc( G, 1, 0); GRAPHinsertArc( G, 1, 5);
   GRAPHinsertArc( G, 2, 4); GRAPHinsertArc( G, 3, 1);
   GRAPHinsertArc( G, 5, 3);

   printf( "listas de adjacencia (note a ordem invertida):\n");
   GRAPHshow( G);

   for (vertex v = 0; v < G->V; ++v) pre[v] = -1;
   cnt = 0;
   dfsR( G, 0);

   printf( "\npre[] =");
   for (vertex v = 0; v < G->V; ++v) printf( " %2d", pre[v]);

   /* Reconstroi a ordem de visita a partir de pre[]. */
   printf( "\nordem de visita:");
   for (int k = 0; k < cnt; ++k)
      for (vertex v = 0; v < G->V; ++v)
         if (pre[v] == k) printf( " %d", v);
   printf( "\n");
   return 0;
}

Overwriting src/dfs_listas.c


In [5]:
compilar_e_rodar('dfs_listas.c')
# Compare com o exemplo anterior: mesmos vertices alcancados, ordem diferente.

listas de adjacencia (note a ordem invertida):
 0:  5  1
 1:  5  0
 2:  4
 3:  1
 4:
 5:  3

pre[] =  0  3 -1  2 -1  1
ordem de visita: 0 5 3 1


### Exercício 1 — quantos vértices são alcançáveis?

Escreva `GRAPHreach(G, s)`, que devolve o **número de vértices alcançáveis** a
partir de `s` (contando o próprio `s`).

Sugestão: reaproveite `dfsR()`. Depois da busca, quantos vértices têm
`pre[v] != -1`? Note que o valor de `cnt` ao final já responde à pergunta — mas
implemente a contagem explicitamente, ela é mais fácil de justificar.

*(Exercício adaptado de Feofiloff, cap. Busca em profundidade.)*

In [6]:
%%writefile src/ex1_reach.c
/* Exercicio 1: numero de vertices alcancaveis a partir de s. */
#include <stdio.h>
#include <stdlib.h>

#define vertex int
#define maxV 100

struct graph { int V; int A; int **adj; };
typedef struct graph *Graph;

static int **MATRIXint( int r, int c, int val) {
   int **m = malloc( r * sizeof (int *));
   for (vertex i = 0; i < r; ++i) m[i] = malloc( c * sizeof (int));
   for (vertex i = 0; i < r; ++i)
      for (vertex j = 0; j < c; ++j) m[i][j] = val;
   return m;
}
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V; G->A = 0; G->adj = MATRIXint( V, V, 0);
   return G;
}
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   if (G->adj[v][w] == 0) { G->adj[v][w] = 1; G->A++; }
}

static int cnt;
static int pre[maxV];

static void dfsR( Graph G, vertex v) {
   pre[v] = cnt++;
   for (vertex w = 0; w < G->V; ++w)
      if (G->adj[v][w] != 0 && pre[w] == -1)
         dfsR( G, w);
}

/* TODO: devolva quantos vertices sao alcancaveis a partir de s (inclusive s).
   Nao se esqueca de reinicializar pre[] e cnt antes da busca. */
int GRAPHreach( Graph G, vertex s) {
   /* implemente aqui */
   return -1;
}

int main( void) {
   Graph G = GRAPHinit( 6);
   GRAPHinsertArc( G, 0, 1); GRAPHinsertArc( G, 0, 5);
   GRAPHinsertArc( G, 1, 0); GRAPHinsertArc( G, 1, 5);
   GRAPHinsertArc( G, 2, 4); GRAPHinsertArc( G, 3, 1);
   GRAPHinsertArc( G, 5, 3);

   /* Testes automaticos: valores esperados no grafo da aula. */
   int esperado[] = {4, 4, 2, 4, 1, 4};
   int ok = 1;
   for (vertex s = 0; s < G->V; ++s) {
      int r = GRAPHreach( G, s);
      printf( "GRAPHreach(G, %d) = %2d  (esperado %d)%s\n",
              s, r, esperado[s], r == esperado[s] ? "" : "   <-- ERRO");
      if (r != esperado[s]) ok = 0;
   }
   printf( ok ? "\ntodos os testes passaram\n" : "\nha testes falhando\n");
   return 0;
}

Overwriting src/ex1_reach.c


In [7]:
compilar_e_rodar('ex1_reach.c')

avisos do compilador:
src/ex1_reach.c:30:13: warning: function 'dfsR' is not needed and will not be emitted [-Wunneeded-internal-declaration]
   30 | static void dfsR( Graph G, vertex v) {
      |             ^~~~
1 warning generated.



GRAPHreach(G, 0) = -1  (esperado 4)   <-- ERRO
GRAPHreach(G, 1) = -1  (esperado 4)   <-- ERRO
GRAPHreach(G, 2) = -1  (esperado 2)   <-- ERRO
GRAPHreach(G, 3) = -1  (esperado 4)   <-- ERRO
GRAPHreach(G, 4) = -1  (esperado 1)   <-- ERRO
GRAPHreach(G, 5) = -1  (esperado 4)   <-- ERRO

ha testes falhando


## 3. Varredura completa e floresta DFS

Uma única busca só enxerga o que é alcançável a partir da raiz. Para visitar
**todos** os vértices, `GRAPHdfs()` percorre `0..V-1` e dispara uma nova busca
sempre que encontra um vértice ainda não visitado.

Cada busca disparada é uma **árvore**; juntas, formam a **floresta DFS**. O vetor
`pai[]` a armazena: `pai[w] == v` quer dizer que a busca chegou a `w` pelo arco
`v-w`, e `pai[v] == v` identifica as raízes.

In [8]:
%%writefile src/floresta.c
/* Varredura completa: pre[], pai[] e a floresta DFS. */
#include <stdio.h>
#include <stdlib.h>

#define vertex int
#define maxV 100

struct graph { int V; int A; int **adj; };
typedef struct graph *Graph;

static int **MATRIXint( int r, int c, int val) {
   int **m = malloc( r * sizeof (int *));
   for (vertex i = 0; i < r; ++i) m[i] = malloc( c * sizeof (int));
   for (vertex i = 0; i < r; ++i)
      for (vertex j = 0; j < c; ++j) m[i][j] = val;
   return m;
}
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V; G->A = 0; G->adj = MATRIXint( V, V, 0);
   return G;
}
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   if (G->adj[v][w] == 0) { G->adj[v][w] = 1; G->A++; }
}

static int cnt;
static int pre[maxV];
static int pai[maxV];

static void dfsR( Graph G, vertex v) {
   pre[v] = cnt++;
   for (vertex w = 0; w < G->V; ++w)
      if (G->adj[v][w] != 0 && pre[w] == -1) {
         pai[w] = v;               /* registra o arco de arvore v-w */
         dfsR( G, w);
      }
}

/* Visita TODOS os vertices, em uma ou mais buscas. */
void GRAPHdfs( Graph G) {
   cnt = 0;
   for (vertex v = 0; v < G->V; ++v) { pre[v] = -1; pai[v] = -1; }
   for (vertex v = 0; v < G->V; ++v)
      if (pre[v] == -1) {
         pai[v] = v;                /* v e raiz de uma nova arvore */
         dfsR( G, v);
      }
}

int main( void) {
   Graph G = GRAPHinit( 6);
   GRAPHinsertArc( G, 0, 1); GRAPHinsertArc( G, 0, 5);
   GRAPHinsertArc( G, 1, 0); GRAPHinsertArc( G, 1, 5);
   GRAPHinsertArc( G, 2, 4); GRAPHinsertArc( G, 3, 1);
   GRAPHinsertArc( G, 5, 3);

   GRAPHdfs( G);

   printf( "v      :");
   for (vertex v = 0; v < G->V; ++v) printf( " %2d", v);
   printf( "\npre[v] :");
   for (vertex v = 0; v < G->V; ++v) printf( " %2d", pre[v]);
   printf( "\npai[v] :");
   for (vertex v = 0; v < G->V; ++v) printf( " %2d", pai[v]);

   int raizes = 0;
   printf( "\n\nraizes  :");
   for (vertex v = 0; v < G->V; ++v)
      if (pai[v] == v) { printf( " %d", v); raizes++; }

   printf( "\narcos de arvore (em ordem de descoberta):");
   for (int k = 0; k < cnt; ++k)
      for (vertex v = 0; v < G->V; ++v)
         if (pre[v] == k && pai[v] != v) printf( " %d-%d", pai[v], v);

   printf( "\n\na floresta tem %d vertices, %d arvores e %d arcos\n",
           G->V, raizes, G->V - raizes);
   return 0;
}

Overwriting src/floresta.c


In [9]:
compilar_e_rodar('floresta.c')
# Saida esperada: duas arvores (raizes 0 e 2); arcos de arvore 0-1, 1-5, 5-3, 2-4.

v      :  0  1  2  3  4  5
pre[v] :  0  1  4  3  5  2
pai[v] :  0  0  2  5  2  1

raizes  : 0 2
arcos de arvore (em ordem de descoberta): 0-1 1-5 5-3 2-4

a floresta tem 6 vertices, 2 arvores e 4 arcos


### Exercício 2 — a versão iterativa, com pilha explícita

Substitua a recursão por uma pilha de vértices. O esqueleto abaixo já traz a
pilha pronta; falta o laço principal:

1. desempilhe um vértice `v`;
2. se `v` já foi visitado, ignore-o e siga (um mesmo vértice pode ser empilhado
   várias vezes antes de ser visitado);
3. visite `v` (isto é, faça `pre[v] = cnt++` e registre-o em `ordem[]`);
4. empilhe todos os vizinhos ainda não visitados, em ordem crescente.

Empilhando em ordem crescente, o `pop` devolve o **maior** primeiro — por isso a
ordem de visita esperada é `0, 5, 3, 1`, e não `0, 1, 5, 3`.

In [10]:
%%writefile src/ex2_pilha.c
/* Exercicio 2: busca em profundidade com pilha explicita. */
#include <stdio.h>
#include <stdlib.h>

#define vertex int
#define maxV 100

struct graph { int V; int A; int **adj; };
typedef struct graph *Graph;

static int **MATRIXint( int r, int c, int val) {
   int **m = malloc( r * sizeof (int *));
   for (vertex i = 0; i < r; ++i) m[i] = malloc( c * sizeof (int));
   for (vertex i = 0; i < r; ++i)
      for (vertex j = 0; j < c; ++j) m[i][j] = val;
   return m;
}
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V; G->A = 0; G->adj = MATRIXint( V, V, 0);
   return G;
}
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   if (G->adj[v][w] == 0) { G->adj[v][w] = 1; G->A++; }
}

/* --- pilha de vertices, pronta para uso --- */
static vertex pilha[maxV * maxV];
static int topo;
static void STACKinit( void)        { topo = 0; }
static int  STACKempty( void)       { return topo == 0; }
static void STACKpush( vertex v)    { pilha[topo++] = v; }
static vertex STACKpop( void)       { return pilha[--topo]; }

static int cnt;
static int pre[maxV];
static vertex ordem[maxV];   /* ordem[k] = k-esimo vertice visitado */

/* TODO: implemente o laco principal descrito no enunciado. */
void GRAPHdfsPilha( Graph G, vertex s) {
   for (vertex v = 0; v < G->V; ++v) pre[v] = -1;
   cnt = 0;
   STACKinit();
   STACKpush( s);
   while (!STACKempty()) {
      /* implemente aqui */
      STACKpop();   /* remova esta linha ao implementar */
   }
}

int main( void) {
   Graph G = GRAPHinit( 6);
   GRAPHinsertArc( G, 0, 1); GRAPHinsertArc( G, 0, 5);
   GRAPHinsertArc( G, 1, 0); GRAPHinsertArc( G, 1, 5);
   GRAPHinsertArc( G, 2, 4); GRAPHinsertArc( G, 3, 1);
   GRAPHinsertArc( G, 5, 3);

   GRAPHdfsPilha( G, 0);

   vertex esperado[] = {0, 5, 3, 1};
   printf( "visitados : %d (esperado 4)\n", cnt);
   printf( "ordem     :");
   for (int k = 0; k < cnt; ++k) printf( " %d", ordem[k]);
   printf( "\nesperado  : 0 5 3 1\n");

   int ok = (cnt == 4);
   for (int k = 0; k < cnt && k < 4; ++k) if (ordem[k] != esperado[k]) ok = 0;
   printf( ok ? "\nteste passou\n" : "\nteste falhando\n");
   return 0;
}

Overwriting src/ex2_pilha.c


In [11]:
compilar_e_rodar('ex2_pilha.c')

visitados : 0 (esperado 4)
ordem     :
esperado  : 0 5 3 1

teste falhando


## 4. Quanto custa a busca

A conta da aula: com matriz de adjacências gastamos $\Theta(V)$ para examinar os
vizinhos de cada vértice, logo $\Theta(V^2)$ no total; com listas gastamos tantos
passos quantos forem os arcos que saem de cada vértice, logo $\Theta(V + A)$.

O programa abaixo instrumenta as duas versões: cada vez que o laço interno dá um
passo — uma célula da matriz olhada, um nó da lista percorrido — um contador é
incrementado. Compare os dois números para o mesmo grafo.

In [12]:
%%writefile src/custo.c
/* Conta os passos do laco interno da DFS nas duas representacoes. */
#include <stdio.h>
#include <stdlib.h>

#define vertex int
#define maxV 100

/* ---------- matriz ---------- */
static int **adjM;
static int Vm;
static long passosM;
static int preM[maxV];
static int cntM;

static void dfsM( vertex v) {
   preM[v] = cntM++;
   for (vertex w = 0; w < Vm; ++w) {
      passosM++;                      /* uma celula da matriz olhada */
      if (adjM[v][w] != 0 && preM[w] == -1) dfsM( w);
   }
}

/* ---------- listas ---------- */
typedef struct node *link;
struct node { vertex w; link next; };
static link *adjL;
static int Vl;
static long passosL;
static int preL[maxV];
static int cntL;

static void dfsL( vertex v) {
   preL[v] = cntL++;
   for (link a = adjL[v]; a != NULL; a = a->next) {
      passosL++;                      /* um no da lista percorrido */
      if (preL[a->w] == -1) dfsL( a->w);
   }
}

int main( void) {
   int V = 6;
   int arcos[][2] = {{0,1},{0,5},{1,0},{1,5},{2,4},{3,1},{5,3}};
   int A = sizeof arcos / sizeof arcos[0];

   Vm = Vl = V;
   adjM = malloc( V * sizeof (int *));
   for (vertex v = 0; v < V; ++v) adjM[v] = calloc( V, sizeof (int));
   adjL = malloc( V * sizeof (link));
   for (vertex v = 0; v < V; ++v) adjL[v] = NULL;

   for (int i = 0; i < A; ++i) {
      vertex v = arcos[i][0], w = arcos[i][1];
      adjM[v][w] = 1;
      link a = malloc( sizeof (struct node));
      a->w = w; a->next = adjL[v]; adjL[v] = a;
   }

   for (vertex v = 0; v < V; ++v) { preM[v] = -1; preL[v] = -1; }
   cntM = cntL = 0; passosM = passosL = 0;
   for (vertex v = 0; v < V; ++v) if (preM[v] == -1) dfsM( v);
   for (vertex v = 0; v < V; ++v) if (preL[v] == -1) dfsL( v);

   printf( "grafo com V = %d e A = %d\n\n", V, A);
   printf( "matriz : %ld passos do laco interno   (V*V = %d)\n", passosM, V*V);
   printf( "listas : %ld passos do laco interno   (A   = %d)\n", passosL, A);
   printf( "\nrazao matriz/listas = %.2f\n", (double) passosM / passosL);
   return 0;
}

Overwriting src/custo.c


In [13]:
compilar_e_rodar('custo.c')
# Note: com a matriz, o numero de passos nao depende de quantos arcos existem;
# com listas, ele e exatamente A.

grafo com V = 6 e A = 7

matriz : 36 passos do laco interno   (V*V = 36)
listas : 7 passos do laco interno   (A   = 7)

razao matriz/listas = 5.14


### Exercício 3 — recuperar um caminho de `s` a `t`

O vetor `pai[]` guarda a floresta DFS. Use-o para escrever
`GRAPHpath(G, s, t)`, que imprime um caminho de `s` a `t` (na ordem correta,
de `s` para `t`) ou avisa que não existe caminho.

Roteiro: rode a busca a partir de `s`; se `pre[t] == -1`, não há caminho; caso
contrário, siga `pai[]` a partir de `t` até chegar em `s`, guardando os vértices
em um vetor, e imprima-o **de trás para frente**.

Lembre-se: o caminho encontrado pela busca em profundidade não é
necessariamente o mais curto.

In [14]:
%%writefile src/ex3_path.c
/* Exercicio 3: recuperar um caminho de s a t usando pai[]. */
#include <stdio.h>
#include <stdlib.h>

#define vertex int
#define maxV 100

struct graph { int V; int A; int **adj; };
typedef struct graph *Graph;

static int **MATRIXint( int r, int c, int val) {
   int **m = malloc( r * sizeof (int *));
   for (vertex i = 0; i < r; ++i) m[i] = malloc( c * sizeof (int));
   for (vertex i = 0; i < r; ++i)
      for (vertex j = 0; j < c; ++j) m[i][j] = val;
   return m;
}
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V; G->A = 0; G->adj = MATRIXint( V, V, 0);
   return G;
}
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   if (G->adj[v][w] == 0) { G->adj[v][w] = 1; G->A++; }
}

static int cnt;
static int pre[maxV];
static int pai[maxV];

static void dfsR( Graph G, vertex v) {
   pre[v] = cnt++;
   for (vertex w = 0; w < G->V; ++w)
      if (G->adj[v][w] != 0 && pre[w] == -1) {
         pai[w] = v;
         dfsR( G, w);
      }
}

/* TODO: imprima um caminho de s a t, ou "nao ha caminho de s a t". */
void GRAPHpath( Graph G, vertex s, vertex t) {
   for (vertex v = 0; v < G->V; ++v) { pre[v] = -1; pai[v] = -1; }
   cnt = 0;
   pai[s] = s;
   dfsR( G, s);
   /* implemente aqui */
   printf( "caminho de %d a %d: (nao implementado)\n", s, t);
}

int main( void) {
   Graph G = GRAPHinit( 6);
   GRAPHinsertArc( G, 0, 1); GRAPHinsertArc( G, 0, 5);
   GRAPHinsertArc( G, 1, 0); GRAPHinsertArc( G, 1, 5);
   GRAPHinsertArc( G, 2, 4); GRAPHinsertArc( G, 3, 1);
   GRAPHinsertArc( G, 5, 3);

   GRAPHpath( G, 0, 3);   /* esperado: 0 1 5 3 */
   GRAPHpath( G, 0, 2);   /* esperado: nao ha caminho */
   GRAPHpath( G, 2, 4);   /* esperado: 2 4 */
   return 0;
}

Overwriting src/ex3_path.c


In [15]:
compilar_e_rodar('ex3_path.c')

caminho de 0 a 3: (nao implementado)
caminho de 0 a 2: (nao implementado)
caminho de 2 a 4: (nao implementado)


## Desafio Final — alcançabilidade em um grafo maior

Um **arquivo de arcos** é um arquivo de texto em que a primeira linha contém
$V$, a segunda contém $A$, e cada uma das $A$ linhas seguintes contém dois
inteiros em $0..V-1$ que descrevem um arco.

Escreva um programa que:

1. leia um arquivo de arcos da entrada padrão e construa o grafo com listas de
   adjacência;
2. faça a busca em profundidade a partir do vértice 0 e imprima quantos e quais
   vértices são alcançáveis;
3. faça a varredura completa `GRAPHdfs()` e imprima quantas árvores tem a
   floresta DFS, com suas raízes;
4. imprima um caminho de 0 até o vértice 10, se existir.

A entrada é o grafo do Exemplo A do capítulo *Grafos* (Feofiloff), o mesmo da
aula anterior: 12 vértices e 16 arcos.

In [16]:
# Arquivo de arcos do Exemplo A: 0-5 0-6 2-0 2-3 3-6 3-10 4-1 5-2 5-10
#                                6-2 7-8 7-11 8-1 8-4 10-3 11-8
arcos = [(0,5),(0,6),(2,0),(2,3),(3,6),(3,10),(4,1),(5,2),
         (5,10),(6,2),(7,8),(7,11),(8,1),(8,4),(10,3),(11,8)]
with open('src/exemploA.txt', 'w') as f:
    f.write('12\n%d\n' % len(arcos))
    for v, w in arcos:
        f.write('%d %d\n' % (v, w))
print(open('src/exemploA.txt').read())

12
16
0 5
0 6
2 0
2 3
3 6
3 10
4 1
5 2
5 10
6 2
7 8
7 11
8 1
8 4
10 3
11 8



In [17]:
%%writefile src/desafio.c
/* Desafio final: alcancabilidade e floresta DFS em um grafo lido da entrada. */
#include <stdio.h>
#include <stdlib.h>

#define vertex int
#define maxV 100

typedef struct node *link;
struct node { vertex w; link next; };
struct graph { int V; int A; link *adj; };
typedef struct graph *Graph;

static link NEWnode( vertex w, link next) {
   link a = malloc( sizeof (struct node));
   a->w = w; a->next = next; return a;
}
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V; G->A = 0;
   G->adj = malloc( V * sizeof (link));
   for (vertex v = 0; v < V; ++v) G->adj[v] = NULL;
   return G;
}
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   for (link a = G->adj[v]; a != NULL; a = a->next)
      if (a->w == w) return;
   G->adj[v] = NEWnode( w, G->adj[v]); G->A++;
}

static int cnt;
static int pre[maxV];
static int pai[maxV];

/* TODO: escreva dfsR() para listas de adjacencia, registrando pai[]. */

/* TODO: leia V, depois A, depois os A pares de vertices, e devolva o grafo. */
Graph GRAPHinputArcs( void) {
   /* implemente aqui */
   return NULL;
}

int main( void) {
   Graph G = GRAPHinputArcs();
   if (G == NULL) { printf( "GRAPHinputArcs() ainda nao implementada\n"); return 1; }

   /* TODO: itens 2 a 4 do enunciado. */

   return 0;
}

Overwriting src/desafio.c


In [18]:
compilar_e_rodar('desafio.c', entrada=open('src/exemploA.txt').read())

avisos do compilador:
src/desafio.c:30:12: warning: unused variable 'cnt' [-Wunused-variable]
   30 | static int cnt;
      |            ^~~
src/desafio.c:31:12: warning: unused variable 'pre' [-Wunused-variable]
   31 | static int pre[maxV];
      |            ^~~
src/desafio.c:32:12: warning: unused variable 'pai' [-Wunused-variable]
   32 | static int pai[maxV];
      |            ^~~
3 warnings generated.



GRAPHinputArcs() ainda nao implementada


### Sua conclusão

Responda aqui (edite esta célula):

- Quantos dos 12 vértices são alcançáveis a partir de 0? ____
- Quantas árvores tem a floresta DFS do Exemplo A, e quais são as raízes? ____
- O caminho de 0 a 10 que a sua busca encontrou é o mais curto possível? Como
  você poderia verificar isso?
- Se o grafo tivesse 10 000 vértices e 30 000 arcos, quantos passos a busca faria
  com listas? E com matriz?

## Referências

- FEOFILOFF, P. *Algoritmos para Grafos (em linguagem C)*. IME-USP, 2020.
  Capítulo [Busca em profundidade](https://www.ime.usp.br/~pf/algoritmos_para_grafos/aulas/dfs.html).
- SEDGEWICK, R. *Algorithms in C, Part 5: Graph Algorithms*. 3. ed. Addison-Wesley, 2002. Cap. 18.
- CORMEN, T. H. et al. *Introduction to Algorithms*. 3. ed. MIT Press, 2009. Seção 22.3.
- TARJAN, R. Depth-First Search and Linear Graph Algorithms. *SIAM J. Comput.*, v. 1, n. 2, p. 146–160, 1972.